
# RSA-Based Hybrid Encryption and Decryption (Part C)

## Pre-Implementation and verification

We assumed the correctness of the textbook RSA property, where $(m^e)^d \pmod n = m$. As long as we got the correctly formatted the random integer r, it is guaranteed it will be safely recovered.

We mapped out everything the reciever would need to rebuild the original message as the sender encrypted it. The receiver uses their private RSA key for decrypting the secret seed ($r$) safely. Because hash functions always give the same output for the same input, the receiver can feed the recovered seed ($r$) and the block number ($i$) into SHA-256, which guarantees they recreate the exact same keystream ($H(i, r)$), which the sender used.

As both the sender and reciever can build the exact same keystream, the decryption should logically follow. By the nature of XOR operation, combining the ciphertext with keystream a second time reverses the encryption and gives back the original file block.

We also identified that files rarely have the size that are exact multiples of 32 B, so we made sure the math would work for those cases as well.

## Code Implementation and Explanation

In this part of the assignment, we implemented a hybrid encryption scheme using RSA for handling key encryption and SHA-256 for securing the actual data.

---

### **Key Generation and Core RSA Functions**

We used the `cryptography.hazmat.primitives` library to generate a 2048-bit RSA key pair, which is considered to be secure. The public exponent was set to 65537, which is a common choice for its computational efficiency.

From the generated key we extracted the components ($n$, $e$, and $d$) and implemented RSA manually using Python’s built-in `pow()` function for modular exponentiation:

- **RSA Encrypt:**  
  $c = m^e \pmod n$

- **RSA Decrypt:**  
  $m = c^d \pmod n$

---

### **Hybrid Encryption Algorithm ($Enc(m; r)$)**

 RSA on itself is too slow for encrypting large files, so we used a hybrid approach: encrypt a small random key with RSA, and use that key to encrypt the actual data.

$$
Enc(m; r) = (RSA(r), H(0, r) \oplus m_0, \dots, H(n, r) \oplus m_n)
$$

The implementation works as follows:

1. **Key Encapsulation:**  
   A random 32-byte (256-bit) value $r$ is generated using `os.urandom(32)` as a one-time session key. We convert it to an integer, encrypt it using RSA, and store it as a 256-byte value (to match the 2048-bit key size).

2. **Data Encapsulation (Stream Cipher):**  
   SHA-256 is used to simulate a stream cipher. The data gets split into 32-byte blocks, and for each block $m_i$, we concatenate the block index $i$ with the key $r$, hash them together, and use the result as a keystream. This keystream is then XORed ($\oplus$) with the data block.

3. **Decryption:**  
   First, we take the first 256 bytes of the ciphertext and decrypt it using the private key $d$ to get back $r$. From there, we generate the same SHA-256 keystream and XOR it with the remaining ciphertext blocks to recover the original file.

*Note: The full Python implementation is included in the attached compressed file.*

---

## Experimental Setup and Benchmarking Methodology

For measurements of the execution time, we have used the Python `timeit` module.

- **Setup:**  
  The benchmarks were run on a **xxx** machine with an **xxx** processor and **xxx** of RAM, running Python **xxx**.
- **Isolation:**  
  To isolate just the crypto operations, all files were read into memory before starting the timer. The tested file sizes were 8, 64, 512, 4096, 32768, 262144, and 2097152 bytes.

- **Statistical Significance:**  
  Each operation (encryption and decryption) was repeated **100 times** for every file size. A file of each size is then encrypted for decryption benchmarking and the measurement is repeated a hundred times as well. We then computed the mean and standard deviation across runs to make sure the results were consistent. The times were measured in seconds and converted to milliseconds for plotting.

---

## Performance Results

The plots below show encryption and decryption times for different file sizes.

### Encryption and Decryption Plot

![RSA Encryption Plot](rsa_public_combined_plot.png)

*Figure: RSA-based encryption times (X-axis: bytes, Y-axis: μs)*

---

*NOTE: Docstring briefs for this part were generated by Claude from the source code and graph plotting function was partly generated by Gemini, mainly the design part.*

## Part E: Results Analysis and Comparisons

### Experimental Setup and Methodology

To make sure our results were reliable, we measured execution time using Python’s `timeit` module.

- **Environment:**  
  The benchmarks were run on a `xxx` system using Python `xxx`.

- **Methodology:**  
  We excluded file I/O (reading files from disk) from measurements so that we only measured the cryptographic computations. Each operation was repeated **100 times** for every file size.

- **Metrics:**  
  The results show the average execution time in **miliseconds**. We also calculated the standard deviation to make sure the results were consistent.

*(Note: Run the code cell above to generate the comparative plots.)*

---

### 1. RSA-Based Encryption vs. Decryption Times

**Observation:**  
For all file sizes, RSA-based decryption is much slower than encryption. Both increase linearly as file size grows, but decryption starts with a much higher initial cost. Left part of each graph clearly shows the fixed overhead of the RSA math: It clearly shows the algorithm contains two different phases.

**Explanation:**  
This difference comes from how RSA works mathematically during the key encapsulation step (encrypting/decrypting the session key $r$). Because the stream cipher phase is almost identical for both encryption and decryption, the two lines scale at the same rate, only separated by a constant gap of the initial decryption step.

- **Encryption ($c = r^e \pmod n$):**  
  The public exponent $e = 65537$ is small and has a simple binary form. This makes the computation very fast using the square-and-multiply algorithm.

- **Decryption ($r = c^d \pmod n$):**  
  The private exponent $d$ is a very large number (close to 2048 bits). Performing modular exponentiation with such a large exponent is much more computationally expensive, which slows down decryption significantly. With smaller file sizes, the orange line (Decryption) sits way above the blue line (Encryption), which visually proves that decrypting with a long private key takes significantly more time than encrypting it.

  The linear part of the left graph proves that the SHA-256 hash loop scales with ($O(N)$) complexity, depending on the file size. At larger scales, the RSA math at the beginning of computation is just a tiny bump and hashing and xoring data takes majority of the time.

---

### 2. AES-Based Encryption vs. RSA-Based Encryption

**Observation:**  
When comparing AES-256 (CTR mode) with our hybrid scheme (RSA-2048 + SHA-256), AES is significantly faster, especially for larger files.

**Explanation:**  
Initial Overhead: The hybrid RSA method starts slower because it first has to do a complex 2048-bit calculation to encrypt the session key $r$ securely. Only after that it can start processing the actual file. In contrast, AES just sets up its encryption and starts computing almost immediately.


- **Data Encapsulation:**  
 Both methods take linearly more time as the file gets bigger, but AES is much faster in practice. It’s highly optimized and often supported directly by the CPU. Our RSA hybrid approach works like a stream cipher using SHA-256, but it has to hash every block of data ($H(i, r) \oplus m_i$), which makes it more computationally expensive per byte compared to AES.